In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML

cc = pd.read_parquet('../data/hm_all_candidate_comparison.parquet')

# Only human rows, only completed
df = cc[
    (cc['metadata.provenance'] == 'HUMAN_CITIZEN') &
    (cc['metadata.status'] == 'COMPLETED')
].copy()

print(f'Loaded {len(df):,} human completed rows')
print(f'{df["question.text"].nunique():,} unique questions across {df["question.topic"].nunique()} topics')

In [ ]:
# Build question index: question text -> (topic, split, n_participants)
q_stats = (
    df.groupby('question.text')
    .agg(
        topic=('question.topic', 'first'),
        split=('question.split', 'first'),
        n=('worker_id', 'count'),
    )
    .sort_values(['topic', 'n'], ascending=[True, False])
    .reset_index()
)

# Dropdown options: "[topic] question text (n responses)"
q_options = [
    (f"[{row['topic']}] {row['question.text']} ({row['n']})", row['question.text'])
    for _, row in q_stats.iterrows()
]

print(f'{len(q_options)} questions available')

In [ ]:
LIKERT_ORDER = ['STRONGLY_AGREE', 'AGREE', 'SOMEWHAT_AGREE', 'NEUTRAL', 'SOMEWHAT_DISAGREE', 'DISAGREE', 'STRONGLY_DISAGREE']
QUALITY_ORDER = ['EXCELLENT_QUALITY', 'GOOD_QUALITY', 'POOR_QUALITY']
QUALITY_EMOJI = {'EXCELLENT_QUALITY': '★★★', 'GOOD_QUALITY': '★★☆', 'POOR_QUALITY': '★☆☆'}
LIKERT_EMOJI = {
    'STRONGLY_AGREE': '●●●●', 'AGREE': '●●●○', 'SOMEWHAT_AGREE': '●●○○',
    'NEUTRAL': '●○○○', 'SOMEWHAT_DISAGREE': '◐○○○', 'DISAGREE': '◑○○○', 'STRONGLY_DISAGREE': '○○○○'
}

def render_question(question_text):
    rows = df[df['question.text'] == question_text].sort_values(['round_id', 'iteration_index', 'worker_id'])
    if rows.empty:
        return HTML('<p>No data</p>')

    meta = rows.iloc[0]
    topic = meta['question.topic']
    split = meta['question.split']
    affirming = meta.get('question.affirming_statement', '')
    negating = meta.get('question.negating_statement', '')

    parts = []
    parts.append(f'''
    <div style="font-family:sans-serif; max-width:900px">
    <h2 style="color:#1a1a2e">{question_text}</h2>
    <p style="color:#555; font-size:0.9em">Topic {topic} &nbsp;|&nbsp; Split: {split} &nbsp;|&nbsp; {len(rows)} responses</p>
    <table style="border-collapse:collapse; width:100%; font-size:0.85em; margin-bottom:12px">
      <tr>
        <td style="padding:4px 8px; color:#27ae60"><b>Affirming:</b> {affirming}</td>
        <td style="padding:4px 8px; color:#c0392b"><b>Negating:</b> {negating}</td>
      </tr>
    </table>
    ''']

    for round_id, round_rows in rows.groupby('round_id'):
        short_id = round_id[:8]
        parts.append(f'<div style="border:1px solid #ddd; border-radius:6px; margin-bottom:16px; overflow:hidden">')
        parts.append(f'<div style="background:#2c3e50; color:white; padding:8px 12px; font-weight:bold">Round {short_id}</div>')

        for iter_idx, iter_rows in round_rows.groupby('iteration_index'):
            parts.append(f'<div style="background:#ecf0f1; padding:6px 12px; font-size:0.8em; color:#555">Iteration {iter_idx}</div>')

            # Top candidate for this iteration (same across rows in this group)
            top_row = iter_rows.iloc[0]
            top_text = top_row.get('top_candidate.text', '')
            top_label = top_row.get('top_candidate.display_label', '')
            agg_method = top_row.get('top_candidate.aggregation.method', '')
            welfare = top_row.get('top_candidate.reward_data.welfare_or_rank', None)
            welfare_str = f' &nbsp; reward={welfare:.2f}' if pd.notna(welfare) else ''

            parts.append(f'''
            <div style="background:#fef9e7; border-left:4px solid #f39c12; margin:8px 12px; padding:8px 10px">
              <b>Top candidate [{top_label}]</b> <span style="color:#888; font-size:0.85em">{agg_method}{welfare_str}</span><br>
              <span style="color:#333">{top_text}</span>
            </div>
            ''')

            # Per-participant rows
            for _, prow in iter_rows.iterrows():
                worker = str(prow['worker_id'])[:8] if pd.notna(prow['worker_id']) else 'anon'
                own_text = prow.get('own_opinion.text', '') or ''
                critique_text = prow.get('critique.text', '') or ''
                status = prow.get('metadata.status', '')

                # Build ratings summary from parallel arrays
                cands = prow.get('candidates.display_label', [])
                agreements = prow.get('ratings.agreement', [])
                qualities = prow.get('ratings.quality', [])
                ranks = prow.get('rankings.numerical_ranks', [])

                ratings_html = ''
                if cands is not None and len(cands) > 0:
                    try:
                        rank_map = {int(r): lbl for lbl, r in zip(cands, ranks)} if len(ranks) == len(cands) else {}
                        cells = []
                        for i, lbl in enumerate(cands):
                            agr = agreements[i] if i < len(agreements) else ''
                            qual = qualities[i] if i < len(qualities) else ''
                            rank_num = list(ranks).index(i) + 1 if i < len(ranks) and i in ranks else ''
                            rank_val = int(ranks[i]) + 1 if i < len(ranks) else '?'
                            is_top = (lbl == top_label)
                            bg = '#d5f5e3' if is_top else '#f9f9f9'
                            cells.append(
                                f'<td style="border:1px solid #ddd; padding:3px 6px; background:{bg}; font-size:0.8em">'
                                f'<b>{lbl}</b> rank#{rank_val}<br>'
                                f'{agr}<br>{QUALITY_EMOJI.get(qual,qual)}</td>'
                            )
                        ratings_html = '<table style="border-collapse:collapse; margin-top:4px"><tr>' + ''.join(cells) + '</tr></table>'
                    except Exception:
                        pass

                skip_critique = (
                    not critique_text or
                    'Dummy participant' in critique_text or
                    'filler value' in critique_text
                )
                critique_section = (
                    f'<div style="margin-top:4px; color:#555"><b>Critique:</b> {critique_text}</div>'
                    if not skip_critique else ''
                )

                parts.append(f'''
                <div style="border-top:1px solid #eee; padding:8px 12px">
                  <div style="font-size:0.78em; color:#888; margin-bottom:4px">worker <code>{worker}</code></div>
                  <div style="color:#222"><b>Opinion:</b> {own_text}</div>
                  {critique_section}
                  {ratings_html}
                </div>
                ''')

        parts.append('</div>')

    parts.append('</div>')
    return HTML(''.join(parts))


# UI
dropdown = widgets.Combobox(
    options=[label for label, _ in q_options],
    description='Question:',
    layout=widgets.Layout(width='100%'),
    style={'description_width': '70px'},
    ensure_option=False,
    placeholder='Type to filter questions...',
)

output = widgets.Output()

def on_change(change):
    label = change['new']
    # find the question text for this label
    match = next((qtext for lbl, qtext in q_options if lbl == label), None)
    if match is None:
        # try substring match
        match = next((qtext for lbl, qtext in q_options if label.lower() in lbl.lower()), None)
    if match:
        with output:
            output.clear_output(wait=True)
            display(render_question(match))

dropdown.observe(on_change, names='value')

# Seed with first question
first_label, first_q = q_options[0]
display(widgets.VBox([dropdown, output]))
with output:
    display(render_question(first_q))
dropdown.value = first_label